In [1]:

from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.tools import tool
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
import requests


In [2]:
load_dotenv()

True

In [3]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=1.0, 
    max_tokens=None,
    timeout=None,
    max_retries=2,
) 

In [4]:
@tool
def get_stock_price(symbol : str)-> dict:
    """
    Fetch latest stock price for a givem symbol(eg. 'AAPL' , 'TSLA')
    using Alpha Vantage API key in the URL.
    """

    url = (
        "https://www.alphavantage.co/query"
        f"?function=GLOBAL_QUOTE&symbol={symbol}&apikey=C9PE94QUEW9VWGFM"
    )
    r = requests.get(url)
    return r.json()

In [5]:

@tool
def purchase_stock(symbol: str, quantity: int) -> dict:
    """
    Simulate purchasing a given quantity of a stock symbol.

    NOTE: This is a mock implementation:
    - No real brokerage API is called.
    - It simply returns a confirmation payload.
    """
    return {
        "status": "success",
        "message": f"Purchase order placed for {quantity} shares of {symbol}.",
        "symbol": symbol,
        "quantity": quantity,
    }


In [6]:
tools = [get_stock_price, purchase_stock]
llm_with_tools = llm.bind_tools(tools)


In [7]:
# 3. State
# -------------------
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]


In [8]:
def chat_node(state: ChatState):
    """LLM node that may answer or request a tool call."""
    messages = state["messages"]
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}

tool_node = ToolNode(tools)

In [10]:
memory = MemorySaver()

In [11]:
# 6. Graph
# -------------------
graph = StateGraph(ChatState)
graph.add_node("chat_node", chat_node)
graph.add_node("tools", tool_node)

graph.add_edge(START, "chat_node")

graph.add_conditional_edges("chat_node", tools_condition)
graph.add_edge("tools", "chat_node")

chatbot = graph.compile(checkpointer=memory)


In [ ]:
# 7. Simple usage example (CLI)
# -------------------
if __name__ == "__main__":
    print("📈 Stock Bot with Tools (get_stock_price, purchase_stock)")
    print("Type 'exit' to quit.\n")

    # thread_id still works with MemorySaver (conversation kept in RAM)
    thread_id = "demo-thread"

    while True:
        user_input = input("You: ")
        if user_input.lower().strip() in {"exit", "quit"}:
            print("Goodbye!")
            break

        # Build initial state for this turn
        state = {"messages": [HumanMessage(content=user_input)]}

        # Run the graph
        result = chatbot.invoke(
            state,
            config={"configurable": {"thread_id": thread_id}},
        )

        # Get the latest message from the assistant
        messages = result["messages"]
        last_msg = messages[-1]
        print(f"Bot: {last_msg.content}\n")

📈 Stock Bot with Tools (get_stock_price, purchase_stock)
Type 'exit' to quit.

Bot: [{'type': 'text', 'text': 'I am sorry, I cannot find the stock price for VI.', 'extras': {'signature': 'CrcCAXLI2nwtc9uywfSYpmiuNf1Z4OxOS+44xv3YUPjyCstJTcxECU0GOLk8f5Y1/0n5D2JGZ1Kc42YxfKK4zX3Y2nP9dizcTwsm5mFeagStvWAxDmVPEcH/81WnygvreiGyGP2M/Zx7yTG/jI3JgRx7KLtCIYE3lhlEpdD1vvlaM0JppEtUh2BUa21kEvLcY++MLJHbm5nBuENscycLvuYAPpuXt9N6d1oLk9qPxlyajeNwDKrl8UMiuzljoD7jE9qgfU0OhqyovZw8u4wEWiiBKQR9crdXB8NciZk7+HAioignzO6xy5/7WUpDjPmxfSrtMwj/nvPDz6DtVtmg6/AuJEG+KSu3oKGhNHP9H4kk4RLwnHpVvtLAYs96j0SsSzGJmVZVRvlMFX0R76OV6M3+NTn4mbpWdMk='}}]

Bot: Are you trying to find the stock price of Zomato? If so, could you please provide its stock symbol?

Bot: The stock price of Apple (AAPL) is 260.33.

Bot: You're welcome! Is there anything else I can assist you with today?



d:\LangGraph\env\Lib\site-packages\langchain_google_genai\chat_models.py:2832: UserWarning: HumanMessage with empty content was removed to prevent API error
  warnings.warn(


Bot: 

Bot: 

Bot: 

Bot: 

